# Workforce Optimization — Multi-Objective with cuOpt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVIDIA/cuopt-examples/blob/main/workforce_optimization/workforce_optimization_multiobjective.ipynb)

The base `workforce_optimization_milp` notebook minimizes labor cost with everything else **hard-constrained** — one objective, one plan. Real staffing weighs several conflicting considerations with **no fixed weighting**: **labor cost**, **coverage / service level**, **fairness** (workload balance), and — with extra data — **overtime cost** and **worker preferences**.

The key move of the `cuopt-multi-objective-exploration` skill: **any hard constraint is a candidate objective.** Promote one to a *parametric* constraint and sweep it to trace the tradeoff (the ε-constraint method). The base model's two hard constraints are exactly such candidates:

- coverage `Σ x[·,s] == required[s]`  → relax to an **objective** ⇒ **cost vs. coverage**
- `Σ x[w,·] ≤ max_shifts`  → **sweep the cap** ⇒ **cost vs. fairness** (turning a constraint into an objective, with no new data)

This notebook battle-tests the skill on a net-new multi-objective MILP: both tradeoffs above, the ε-constraint-vs-weighted-sum **gotcha**, a `time_limit` on every solve, and the honest **"no duals for a MILP"** note. Overtime and preferences are further candidate objectives (they need overtime-rate / preference data) and are left as extensions.

> **Requirements.** cuOpt needs **Linux + an NVIDIA GPU** (Colab: *Runtime → Change runtime type → GPU*).

## Environment Setup

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print(out.stdout.strip() or "no nvidia-smi output")
except FileNotFoundError:
    print("No NVIDIA GPU detected - cuOpt cannot run. In Colab: Runtime -> Change runtime type -> GPU.")

In [ ]:
# Uncomment if cuOpt is not already installed (e.g., Google Colab):
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu12

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cuopt.linear_programming.problem import Problem, VType, sense, LinearExpression
from cuopt.linear_programming.solver_settings import SolverSettings
print("Imports ready")

## Problem Data

Same workers, shifts, pay, and availability as the base `workforce_optimization_milp` notebook.

In [ ]:
shift_requirements = {
    "Mon1": 3, "Tue2": 2, "Wed3": 4, "Thu4": 2, "Fri5": 5, "Sat6": 3, "Sun7": 4,
    "Mon8": 2, "Tue9": 2, "Wed10": 3, "Thu11": 4, "Fri12": 5, "Sat13": 7, "Sun14": 5,
}
worker_pay = {"Amy": 10, "Bob": 12, "Cathy": 10, "Dan": 8, "Ed": 8, "Fred": 9, "Gu": 11}
availability = {
    "Amy":   ["Tue2","Wed3","Fri5","Sun7","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Bob":   ["Mon1","Tue2","Fri5","Sat6","Mon8","Thu11","Sat13","Sun14"],
    "Cathy": ["Wed3","Thu4","Fri5","Sun7","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Dan":   ["Tue2","Wed3","Fri5","Sat6","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
    "Ed":    ["Mon1","Tue2","Wed3","Thu4","Fri5","Sun7","Mon8","Tue9","Thu11","Sat13","Sun14"],
    "Fred":  ["Mon1","Tue2","Wed3","Sat6","Mon8","Tue9","Fri12","Sat13","Sun14"],
    "Gu":    ["Mon1","Tue2","Wed3","Fri5","Sat6","Sun7","Mon8","Tue9","Wed10","Thu11","Fri12","Sat13","Sun14"],
}
pairs = [(w, s) for w, shifts in availability.items() for s in shifts]
TOTAL_REQUIRED = sum(shift_requirements.values())
print(f"{len(worker_pay)} workers, {len(shift_requirements)} shifts, {len(pairs)} feasible (worker,shift) pairs")
print(f"Total required coverage (all shifts fully staffed): {TOTAL_REQUIRED}")

## Two objectives, no fixed priority — the Pareto pattern

- **Minimize** total labor cost = Σ pay[w]·x[w,s]
- **Maximize** coverage = Σ x[w,s]   (with `assigned[s] ≤ required[s]` — no overstaffing, which keeps coverage linear)

These conflict (more coverage costs more) and there's no agreed weighting, so we trace the frontier instead of committing to one. Each point is one cuOpt MILP solve. The helper below builds the model once and supports either an **ε-constraint** (a coverage floor) or a **weighted-sum** objective, with a `time_limit` per solve (the skill's practical note: bound each MILP solve).

In [ ]:
def solve_workforce(coverage_floor=None, weight_lambda=None, time_limit=10.0):
    """Build + solve one workforce MILP point.
    - coverage_floor: if set, add constraint (total coverage >= floor)  -> epsilon-constraint
    - weight_lambda:  if set, objective = cost - lambda*coverage          -> weighted-sum
    - default (both None): minimize cost only.
    Returns dict(cost, coverage, status).
    """
    prob = Problem("workforce_mo")
    x = {p: prob.addVariable(name=f"{p[0]}_{p[1]}", vtype=VType.INTEGER, lb=0.0, ub=1.0) for p in pairs}

    # Objective
    obj = LinearExpression([], [], 0.0)
    for (w, s), var in x.items():
        coef = worker_pay[w] - (weight_lambda if weight_lambda is not None else 0.0)
        if coef != 0:
            obj += var * coef
    prob.setObjective(obj, sense.MINIMIZE)

    # No overstaffing: assigned[s] <= required[s]
    for s, req in shift_requirements.items():
        e = LinearExpression([], [], 0.0)
        has = False
        for (w, s2), var in x.items():
            if s2 == s:
                e += var; has = True
        if has:
            prob.addConstraint(e <= req, name=f"cap_{s}")

    # epsilon-constraint: total coverage floor
    if coverage_floor is not None:
        cov = LinearExpression([], [], 0.0)
        for var in x.values():
            cov += var
        prob.addConstraint(cov >= float(coverage_floor), name="coverage_floor")

    settings = SolverSettings()
    settings.set_parameter("time_limit", float(time_limit))
    settings.set_parameter("log_to_console", False)
    prob.solve(settings)

    status = prob.Status.name
    if status not in ("Optimal", "FeasibleFound"):
        return {"cost": None, "coverage": None, "status": status}
    sel = [(w, s) for (w, s), var in x.items() if var.getValue() > 0.5]
    cost = sum(worker_pay[w] for (w, s) in sel)
    coverage = len(sel)
    return {"cost": cost, "coverage": coverage, "status": status}

### Step 1 — anchor the objectives (payoff table)

The skill's first step: solve each objective alone to get the achievable ranges. Minimum cost is trivially 0 (assign no one). Maximum coverage is what the workforce can actually staff given availability and the per-shift caps — found by maximizing coverage (i.e. minimizing −coverage via a large λ).

In [ ]:
# Max achievable coverage: large lambda makes every (pay - lambda) negative, so the solver covers all it can.
anchor = solve_workforce(weight_lambda=max(worker_pay.values()) + 1.0)
COVERAGE_MAX = anchor["coverage"]
print(f"Max achievable coverage: {COVERAGE_MAX} of {TOTAL_REQUIRED} required  (cost at full coverage: ${anchor['cost']})")
print(f"Coverage ranges over [0, {COVERAGE_MAX}]; cost over [0, {anchor['cost']}].")

### Steps 2–3 — ε-constraint sweep, then filter dominated

Minimize cost subject to `coverage ≥ ε`, sweeping ε across the coverage range. This is the skill's preferred method for MILP because it reaches the **whole** frontier, including unsupported points.

In [ ]:
eps_grid = list(range(0, COVERAGE_MAX + 1))   # coverage floors 0..max
eps_points = []
for eps in eps_grid:
    r = solve_workforce(coverage_floor=eps)
    if r["status"] in ("Optimal", "FeasibleFound") and r["cost"] is not None:
        eps_points.append((r["coverage"], r["cost"]))

def non_dominated(points):
    """points = list of (coverage, cost); maximize coverage, minimize cost."""
    out = []
    for (cov, cost) in points:
        if not any((c2 >= cov and k2 <= cost and (c2 > cov or k2 < cost)) for (c2, k2) in points):
            out.append((cov, cost))
    return sorted(set(out))

frontier = non_dominated(eps_points)
print(f"epsilon-constraint solves: {len(eps_points)} | non-dominated frontier points: {len(frontier)}")
for cov, cost in frontier:
    print(f"  coverage {cov:2d}/{TOTAL_REQUIRED}  ->  min cost ${cost}")

### Weighted-sum, and the gotcha

Sweep λ in `minimize Σ(pay − λ)·x`. Each λ implicitly weights cost against coverage. The skill's warning: on a MILP, weighted-sum only returns **supported** (convex-hull) points — it cannot produce efficient points sitting in a non-convex dent, no matter the weight. We compute both and compare which efficient points each method recovers.

In [ ]:
lam_grid = np.linspace(0.0, max(worker_pay.values()) + 1.0, 40)
ws_points = []
for lam in lam_grid:
    r = solve_workforce(weight_lambda=float(lam))
    if r["status"] in ("Optimal", "FeasibleFound") and r["cost"] is not None:
        ws_points.append((r["coverage"], r["cost"]))

ws_frontier = non_dominated(ws_points)
frontier_set = set(frontier)
ws_set = set(ws_frontier)
missed_by_ws = sorted(frontier_set - ws_set)      # efficient points epsilon-constraint found but weighted-sum did not
print(f"weighted-sum distinct non-dominated points: {len(ws_set)}")
print(f"epsilon-constraint non-dominated points:    {len(frontier_set)}")
print(f"Efficient points weighted-sum MISSED (unsupported): {len(missed_by_ws)}")
for cov, cost in missed_by_ws:
    print(f"  coverage {cov}/{TOTAL_REQUIRED}, cost ${cost}  - reachable by epsilon-constraint, not by any weight")
if not missed_by_ws:
    print("  (none on this small instance - see notes: the gap is problem-dependent; the method guarantees completeness regardless)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
if eps_points:
    ec = np.array(sorted(frontier))
    ax.plot(ec[:, 0], ec[:, 1], "o-", color="navy", lw=1.6, label=f"epsilon-constraint frontier ({len(frontier)})")
if ws_frontier:
    wc = np.array(sorted(ws_frontier))
    ax.scatter(wc[:, 0], wc[:, 1], s=90, facecolor="none", edgecolor="darkorange",
               linewidth=1.8, zorder=3, label=f"weighted-sum points ({len(ws_frontier)})")
if missed_by_ws:
    mc = np.array(missed_by_ws)
    ax.scatter(mc[:, 0], mc[:, 1], s=160, marker="x", color="crimson", zorder=4,
               label=f"missed by weighted-sum ({len(missed_by_ws)})")
ax.set_xlabel("Coverage (shifts staffed)"); ax.set_ylabel("Labor cost ($)")
ax.set_title("Workforce: cost vs coverage Pareto frontier (cuOpt MILP)")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Step 4 — read the frontier

The skill's interpretation step. Quote the **exchange rate** ($ per extra shift covered) between adjacent points, flag the **knee**, and leave the choice to the planner — don't collapse the frontier to one "best" plan.

In [ ]:
fr = sorted(frontier)
print("Marginal cost of coverage along the frontier:")
for i in range(1, len(fr)):
    dcov = fr[i][0] - fr[i-1][0]
    dcost = fr[i][1] - fr[i-1][1]
    rate = (dcost / dcov) if dcov else float("nan")
    print(f"  coverage {fr[i-1][0]:2d} -> {fr[i][0]:2d}:  +${dcost:>3} for +{dcov} shift(s)  (~${rate:.1f}/shift)")
print("\nThe planner picks the coverage level worth its marginal cost. No single 'best' - it's a choice.")

## Second tradeoff: cost vs. fairness (promote the max-shifts constraint)

The base notebook *fixed* `max_shifts_per_worker = 4` — a hard constraint. The skill says: that cap is a candidate objective. Keep **full coverage** hard, then **sweep the cap**: a tighter cap spreads work more evenly (fairer) but costs more (you need more, or pricier, workers). Sweeping the cap turns the constraint into the fairness axis — the same ε-constraint mechanic, a structurally different tradeoff.

In [ ]:
def solve_workforce_fairness(max_shifts, time_limit=10.0):
    """Full coverage (hard) + per-worker cap <= max_shifts; minimize cost. Sweeping max_shifts
    turns the base notebook's fixed 'max 4 shifts' constraint into a fairness objective."""
    prob = Problem("workforce_fairness")
    x = {p: prob.addVariable(name=f"{p[0]}_{p[1]}", vtype=VType.INTEGER, lb=0.0, ub=1.0) for p in pairs}
    obj = LinearExpression([], [], 0.0)
    for (w, s), var in x.items():
        if worker_pay[w] != 0:
            obj += var * worker_pay[w]
    prob.setObjective(obj, sense.MINIMIZE)
    # Full coverage (hard, == required)
    for s, req in shift_requirements.items():
        e = LinearExpression([], [], 0.0); has = False
        for (w, s2), var in x.items():
            if s2 == s:
                e += var; has = True
        if has:
            prob.addConstraint(e == req, name=f"cover_{s}")
    # Fairness lever: each worker works at most max_shifts
    for w in worker_pay:
        e = LinearExpression([], [], 0.0); has = False
        for (w2, s), var in x.items():
            if w2 == w:
                e += var; has = True
        if has:
            prob.addConstraint(e <= float(max_shifts), name=f"maxshifts_{w}")
    settings = SolverSettings()
    settings.set_parameter("time_limit", float(time_limit))
    settings.set_parameter("log_to_console", False)
    prob.solve(settings)
    st = prob.Status.name
    if st not in ("Optimal", "FeasibleFound"):
        return {"max_shifts": max_shifts, "cost": None, "status": st}
    sel = [(w, s) for (w, s), var in x.items() if var.getValue() > 0.5]
    busiest = max((sum(1 for (w2, s) in sel if w2 == w) for w in worker_pay), default=0)
    return {"max_shifts": max_shifts, "cost": sum(worker_pay[w] for (w, s) in sel),
            "busiest": busiest, "status": st}

In [ ]:
# Sweep the per-worker cap from loose (cheap) to tight (fair). Tighter -> fairer but costlier; too tight -> infeasible.
cap_grid = list(range(len(shift_requirements), 0, -1))   # 14 down to 1
fair_pts = []
for cap in cap_grid:
    r = solve_workforce_fairness(cap)
    if r["cost"] is not None:
        fair_pts.append((r["max_shifts"], r["cost"], r["busiest"]))
        tag = ""
    else:
        tag = f"  (infeasible at cap={cap}: cannot fully cover with everyone capped this low)"
    print(f"max_shifts cap {cap:2d}: " + (f"min cost ${r['cost']}, busiest worker {r['busiest']} shifts" if r['cost'] is not None else f"INFEASIBLE") + tag)

fig, ax = plt.subplots(figsize=(8, 5))
if fair_pts:
    fp = np.array([(c, k) for (c, k, b) in fair_pts])
    ax.plot(fp[:, 0], fp[:, 1], "o-", color="seagreen", lw=1.6)
    ax.invert_xaxis()   # left = fairer (tighter cap)
ax.set_xlabel("Max shifts allowed per worker  (left = fairer)")
ax.set_ylabel("Labor cost ($) at full coverage")
ax.set_title("Workforce: cost vs fairness (sweeping the max-shifts constraint)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("\nThe planner reads the price of fairness: each step tighter on the cap costs $X more at full coverage.")

## Notes (honest)

- **Synthetic data** — the same toy roster as the base notebook; this demonstrates the *method*, not a real staffing study.
- **Optimal to the gap, within the time limit** — each point is solved by cuOpt's MILP solver under a `time_limit`; points are optimal to the solver's gap, not certified global optima unless it returns `Optimal` at a zero gap.
- **ε-constraint vs weighted-sum** — ε-constraint reaches the complete frontier by construction; weighted-sum only returns supported (convex-hull) points. Whether *this* small instance actually exhibits unsupported points is empirical (the cell above reports it); the completeness guarantee holds regardless of instance.
- **No duals for a MILP** — unlike the continuous portfolio QP (see `portfolio_optimization/`), an integer program has no constraint duals/shadow prices; read the marginal cost of coverage from the frontier itself, as above.

This notebook reproduces the `cuopt-multi-objective-exploration` skill end-to-end on cuOpt's own workforce MILP.